### **Teoría de Autómatas y Lenguajes Formales**
#### Prácticas computacionales de "Análisis sintáctico predictivo y Representación y análisis semántico"

Instrucciones para los Ejercicios

1. **Trabajo en Grupo:**
   - Los ejercicios deben ser resueltos y entregados en grupo.
   - La cantidad de integrantes por grupo será definida el día de la actividad, así como la fecha límite para la entrega.

2. **Uso de Google Colab y Compartir:**
   - Este notebook debe ser copiado al GitHub o Google Drive de alguno de los integrantes del grupo.
   - El grupo será responsable de programar las soluciones, realizar las pruebas y enviar el trabajo final al profesor.

3. **Implementación de los Ejercicios:**
   - Cada ejercicio debe ser implementado de manera que cumpla con los objetivos específicos descritos en cada problema.
   - El código debe devolver claramente la información calculada de acuerdo a lo solicitado.

4. **Calidad del Código:**
   - El código debe ejecutarse sin errores.
   - Es obligatorio incluir **comentarios explicativos** para describir las ideas y conceptos implícitos en el código, facilitando la comprensión de su lógica.

5. **Envío del Trabajo:**
   - Una vez completado, el notebook debe ser enviado a través de Moodle.
   - En caso de dudas, pueden contactarme por correo electrónico a **marcelo.danesi@utec.edu.uy**.

6. **Orientaciones Adicionales:**
   - Asegúrense de que todas las celdas de código hayan sido ejecutadas antes de enviar.
   - Incluyan el nombre completo y correo electrónico de todos los integrantes al inicio del notebook.
   - Si utilizan referencias externas, menciónenlas de forma adecuada.

¡Buena suerte y aprovechen la práctica para consolidar los conceptos de teoría de autómatas y lenguajes formales!

#### **Ejemplo guiado: Cálculo automático de FIRST, FOLLOW y tabla LL(1)**

En esta clase vamos analizar cómo se puede implementar un sistema automático en Python que:

1. Calcule los conjuntos **FIRST** y **FOLLOW** para una gramática.
2. Construya la **tabla de análisis predictivo LL(1)** correspondiente.
3. Use esa tabla para simular el comportamiento de un parser predictivo.

Esto nos permitirá verificar, de forma automatizada, si una palabra pertenece o no al lenguaje generado por una gramática.

---

Vamos a trabajar con la siguiente gramática, que fue discutida en clase y es adecuada para el análisis LL(1):

$$
\begin{aligned}
S &\rightarrow aABb \\
A &\rightarrow c \mid \varepsilon \\
B &\rightarrow d \mid \varepsilon
\end{aligned}
$$

Esta gramática genera palabras como `"ab"`, `"acb"`, `"adb"`, `"acdb"`, etc.

El código Python que sigue a continuación realiza tres tareas:

1. Recibe una gramática en forma de diccionario.
2. Calcula de manera recursiva los conjuntos **FIRST** y **FOLLOW** para cada no terminal.
3. Imprime los conjuntos calculados para su inspección.

**Leé los comentarios del código con atención antes de ejecutarlo.**


In [ ]:
# Gramática de ejemplo (LL(1))
gramatica = {
    "S": [["a", "A", "B", "b"]],
    "A": [["c"], []],     # [] representa ε
    "B": [["d"], []]
}

from collections import defaultdict

# Cálculo de FIRST
def calcular_first(gramatica):
    first = defaultdict(set)
    cambio = True
    while cambio:
        cambio = False
        for nt, producciones in gramatica.items():
            for prod in producciones:
                for simbolo in prod:
                    if simbolo not in gramatica:  # Terminal
                        if simbolo not in first[nt]:
                            first[nt].add(simbolo)
                            cambio = True
                        break
                    else:  # No terminal
                        nuevos = first[simbolo] - {"ε"}
                        if not nuevos.issubset(first[nt]):
                            first[nt].update(nuevos)
                            cambio = True
                        if "ε" not in first[simbolo]:
                            break
                else:
                    if "ε" not in first[nt]:
                        first[nt].add("ε")
                        cambio = True
    return dict(first)

# Cálculo de FOLLOW
def calcular_follow(gramatica, first, simbolo_inicial):
    follow = defaultdict(set)
    follow[simbolo_inicial].add("$")
    cambio = True
    while cambio:
        cambio = False
        for nt, producciones in gramatica.items():
            for prod in producciones:
                for i, simbolo in enumerate(prod):
                    if simbolo in gramatica:
                        siguiente = prod[i+1:]
                        first_siguiente = set()
                        for s in siguiente:
                            if s in gramatica:
                                first_siguiente |= (first[s] - {"ε"})
                                if "ε" not in first[s]:
                                    break
                            else:
                                first_siguiente.add(s)
                                break
                        else:
                            first_siguiente.add("ε")

                        if not first_siguiente - {"ε"} <= follow[simbolo]:
                            follow[simbolo] |= (first_siguiente - {"ε"})
                            cambio = True

                        if "ε" in first_siguiente or not siguiente:
                            if not follow[nt] <= follow[simbolo]:
                                follow[simbolo] |= follow[nt]
                                cambio = True
    return dict(follow)

# Mostrar resultados
first = calcular_first(gramatica)
follow = calcular_follow(gramatica, first, "S")

print("📘 FIRST:")
for nt, conj in first.items():
    print(f"  {nt}: {conj}")

print("\n📙 FOLLOW:")
for nt, conj in follow.items():
    print(f"  {nt}: {conj}")


📘 FIRST:
  S: {'a'}
  A: {'ε', 'c'}
  B: {'d', 'ε'}

📙 FOLLOW:
  S: {'$'}
  A: {'d', 'b'}
  B: {'b'}


##### 📐 Construcción de la tabla LL(1)

Una vez calculados los conjuntos **FIRST** y **FOLLOW**, podemos construir la tabla de análisis predictivo LL(1). Esta tabla nos dice qué producción aplicar cuando el parser encuentra un no terminal en la pila y un símbolo en la entrada.

📌 Vamos a usar la siguiente regla:
- Para cada producción $A \to \alpha$:
  - Para cada símbolo $a$ en $FIRST(\alpha)$, agregamos $A \to \alpha$ en la celda $[A, a]$.
  - Si $\varepsilon \in FIRST(\alpha)$, entonces agregamos $A \to \varepsilon$ en todas las celdas $[A, b]$ para cada $b \in FOLLOW(A)$.


In [ ]:
# Esta función recorre todas las producciones de la gramática
# y extrae los símbolos terminales (los que no son no terminales).
# También agrega el símbolo $ al final, usado como marcador de fin de entrada.

def obtener_terminales(gramatica):
    terminales = set()
    for prods in gramatica.values():
        for prod in prods:
            for simbolo in prod:
                if simbolo not in gramatica:
                    terminales.add(simbolo)
    terminales.add("$")
    return sorted(terminales)


from collections import defaultdict

# Esta función construye la tabla LL(1) y detecta conflictos:
# Si una celda ya tiene una producción y se intenta colocar otra distinta,
# se registra como un conflicto.

def construir_tabla_ll1(gramatica, first, follow):
    tabla = defaultdict(dict)
    conflictos = []

    for nt, producciones in gramatica.items():
        for prod in producciones:
            first_prod = set()

            # Obtener el FIRST de esta producción
            for simbolo in prod:
                if simbolo not in gramatica:
                    first_prod.add(simbolo)
                    break
                else:
                    first_prod |= (first[simbolo] - {"ε"})
                    if "ε" not in first[simbolo]:
                        break
            else:
                first_prod.add("ε")

            # Parte 1: insertar la producción en la tabla según su FIRST
            for terminal in first_prod - {"ε"}:
                if terminal in tabla[nt]:
                    conflictos.append((nt, terminal, tabla[nt][terminal], prod))
                tabla[nt][terminal] = prod

            # Parte 2: si ε está en FIRST, usar FOLLOW para completar
            if "ε" in first_prod:
                for terminal in follow[nt]:
                    if terminal in tabla[nt]:
                        conflictos.append((nt, terminal, tabla[nt][terminal], prod))
                    tabla[nt][terminal] = []  # Representa A → ε

    return tabla, conflictos


# Generar la tabla
tabla_ll1, conflictos = construir_tabla_ll1(gramatica, first, follow)

# Obtener la lista de terminales de toda la gramática
terminales = obtener_terminales(gramatica)

# Mostrar encabezado
print("📊 Tabla LL(1):\n")
print(f"{'NT':<5}", "\t".join(terminales))

# Mostrar cada fila de la tabla
for nt in gramatica:
    fila = []
    for t in terminales:
        prod = tabla_ll1[nt].get(t, "")
        if prod == []:
            fila.append("ε")  # Producción vacía
        elif prod:
            fila.append(" ".join(prod))  # Mostrar producción como string
        else:
            fila.append("")  # Celda vacía
    print(f"{nt:<5}", "\t".join(fila))

# Reportar conflictos si los hay
if conflictos:
    print("\n⚠️ Conflictos detectados: la gramática NO es LL(1).")
    for nt, terminal, existente, nuevo in conflictos:
        print(f"  - Conflicto en [{nt}, {terminal}]: ya existe {existente}, se intenta agregar {nuevo}")
else:
    print("\n✅ Gramática confirmada como LL(1): no se detectaron conflictos en la tabla.")


📊 Tabla LL(1):

NT    $	a	b	c	d
S     	a A B b			
A     		ε	c	ε
B     		ε		d

✅ Gramática confirmada como LL(1): no se detectaron conflictos en la tabla.


##### **🤖 Simulación paso a paso del parser LL(1)**

Ahora que tenemos la tabla de análisis predictivo construida, vamos a implementar un parser que simula el comportamiento del análisis sintáctico predictivo.

Este parser:

- Usa una **pila** que comienza con el símbolo inicial ($S$) y el símbolo de fin de entrada ($\$$).
- Lee la **entrada** palabra por palabra, agregando también un $\$$ al final.
- Consulta la **tabla LL(1)** para decidir qué producción aplicar en cada paso.
- Acepta si logra vaciar la pila y consumir toda la entrada de manera sincronizada.

Se mostrará el contenido de la pila y la entrada restante en cada paso.


In [ ]:
def parse_ll1(entrada, tabla, simbolo_inicial):
    pila = ["$", simbolo_inicial]
    entrada = list(entrada) + ["$"]
    pasos = []

    while pila:
        tope = pila.pop()
        actual = entrada[0]

        pasos.append((list(pila), list(entrada)))

        if tope == actual:
            entrada.pop(0)
        elif tope not in tabla:
            return False, pasos
        elif actual in tabla[tope]:
            produccion = tabla[tope][actual]
            for simbolo in reversed(produccion):
                if simbolo != "":
                    pila.append(simbolo)
        else:
            return False, pasos

        if pila == ["$"] and entrada == ["$"]:
            pasos.append(([], ["$"]))
            return True, pasos
    return False, pasos


# Probar una palabra
palabra = list("acdb")  # o list("ab"), list("adb"), etc.
acepta, pasos = parse_ll1(palabra, tabla_ll1, "S")

if acepta:
    print("✅ Palabra aceptada por el parser LL(1).\n")
    for i, (pila, entrada) in enumerate(pasos):
        print(f"Paso {i+1}: Pila = {pila}, Entrada = {entrada}")
else:
    print("❌ Palabra no aceptada.")
    for i, (pila, entrada) in enumerate(pasos):
        print(f"Paso {i+1}: Pila = {pila}, Entrada = {entrada}")


✅ Palabra aceptada por el parser LL(1).

Paso 1: Pila = ['$'], Entrada = ['a', 'c', 'd', 'b', '$']
Paso 2: Pila = ['$', 'b', 'B', 'A'], Entrada = ['a', 'c', 'd', 'b', '$']
Paso 3: Pila = ['$', 'b', 'B'], Entrada = ['c', 'd', 'b', '$']
Paso 4: Pila = ['$', 'b', 'B'], Entrada = ['c', 'd', 'b', '$']
Paso 5: Pila = ['$', 'b'], Entrada = ['d', 'b', '$']
Paso 6: Pila = ['$', 'b'], Entrada = ['d', 'b', '$']
Paso 7: Pila = ['$'], Entrada = ['b', '$']
Paso 8: Pila = [], Entrada = ['$']


#### **Ejercicio 1: Reconocimiento de palabras válidas**

**Descripción:**  
En este ejercicio vamos utilizar el parser LL(1) construido anteriormente para verificar si ciertas palabras pertenecen al lenguaje definido por la siguiente gramática:

**Gramática LL(1):**  
$$
\begin{aligned}
S &\rightarrow aABb \\
A &\rightarrow c \mid \varepsilon \\
B &\rightarrow d \mid \varepsilon
\end{aligned}
$$

**Instrucciones:**
- Utilizá la función `parse_ll1` para analizar las siguientes palabras:
  - `"ab"`
  - `"acb"`
  - `"adb"`
  - `"acdb"`
  - `"a"`
  - `"abbb"`

**Objetivo:**  
Observar cómo el parser se comporta con cada palabra, analizar el contenido de la pila y de la entrada en cada paso, y reflexionar sobre por qué algunas son aceptadas y otras no.

In [ ]:
# Gramática de ejemplo (LL(1))
gramatica = {
    "S": [["a", "A", "B", "b"]],
    "A": [["c"], []],     # [] representa ε
    "B": [["d"], []]
}

from collections import defaultdict

# Cálculo de FIRST
def calcular_first(gramatica):
    first = defaultdict(set)
    cambio = True
    while cambio:
        cambio = False
        for nt, producciones in gramatica.items():
            for prod in producciones:
                for simbolo in prod:
                    if simbolo not in gramatica:  # Terminal
                        if simbolo not in first[nt]:
                            first[nt].add(simbolo)
                            cambio = True
                        break
                    else:  # No terminal
                        nuevos = first[simbolo] - {"ε"}
                        if not nuevos.issubset(first[nt]):
                            first[nt].update(nuevos)
                            cambio = True
                        if "ε" not in first[simbolo]:
                            break
                else:
                    if "ε" not in first[nt]:
                        first[nt].add("ε")
                        cambio = True
    return dict(first)

# Cálculo de FOLLOW
def calcular_follow(gramatica, first, simbolo_inicial):
    follow = defaultdict(set)
    follow[simbolo_inicial].add("$")
    cambio = True
    while cambio:
        cambio = False
        for nt, producciones in gramatica.items():
            for prod in producciones:
                for i, simbolo in enumerate(prod):
                    if simbolo in gramatica:
                        siguiente = prod[i+1:]
                        first_siguiente = set()
                        for s in siguiente:
                            if s in gramatica:
                                first_siguiente |= (first[s] - {"ε"})
                                if "ε" not in first[s]:
                                    break
                            else:
                                first_siguiente.add(s)
                                break
                        else:
                            first_siguiente.add("ε")

                        if not first_siguiente - {"ε"} <= follow[simbolo]:
                            follow[simbolo] |= (first_siguiente - {"ε"})
                            cambio = True

                        if "ε" in first_siguiente or not siguiente:
                            if not follow[nt] <= follow[simbolo]:
                                follow[simbolo] |= follow[nt]
                                cambio = True
    return dict(follow)

# Mostrar resultados
first = calcular_first(gramatica)
follow = calcular_follow(gramatica, first, "S")

print("📘 FIRST:")
for nt, conj in first.items():
    print(f"  {nt}: {conj}")

print("\n📙 FOLLOW:")
for nt, conj in follow.items():
    print(f"  {nt}: {conj}")

# Esta función recorre todas las producciones de la gramática
# y extrae los símbolos terminales (los que no son no terminales).
# También agrega el símbolo $ al final, usado como marcador de fin de entrada.

def obtener_terminales(gramatica):
    terminales = set()
    for prods in gramatica.values():
        for prod in prods:
            for simbolo in prod:
                if simbolo not in gramatica:
                    terminales.add(simbolo)
    terminales.add("$")
    return sorted(terminales)


from collections import defaultdict

# Esta función construye la tabla LL(1) y detecta conflictos:
# Si una celda ya tiene una producción y se intenta colocar otra distinta,
# se registra como un conflicto.

def construir_tabla_ll1(gramatica, first, follow):
    tabla = defaultdict(dict)
    conflictos = []

    for nt, producciones in gramatica.items():
        for prod in producciones:
            first_prod = set()

            # Obtener el FIRST de esta producción
            for simbolo in prod:
                if simbolo not in gramatica:
                    first_prod.add(simbolo)
                    break
                else:
                    first_prod |= (first[simbolo] - {"ε"})
                    if "ε" not in first[simbolo]:
                        break
            else:
                first_prod.add("ε")

            # Parte 1: insertar la producción en la tabla según su FIRST
            for terminal in first_prod - {"ε"}:
                if terminal in tabla[nt]:
                    conflictos.append((nt, terminal, tabla[nt][terminal], prod))
                tabla[nt][terminal] = prod

            # Parte 2: si ε está en FIRST, usar FOLLOW para completar
            if "ε" in first_prod:
                for terminal in follow[nt]:
                    if terminal in tabla[nt]:
                        conflictos.append((nt, terminal, tabla[nt][terminal], prod))
                    tabla[nt][terminal] = []  # Representa A → ε

    return tabla, conflictos


# Generar la tabla
tabla_ll1, conflictos = construir_tabla_ll1(gramatica, first, follow)

# Obtener la lista de terminales de toda la gramática
terminales = obtener_terminales(gramatica)

# Mostrar encabezado
print("📊 Tabla LL(1):\n")
print(f"{'NT':<5}", "\t".join(terminales))

# Mostrar cada fila de la tabla
for nt in gramatica:
    fila = []
    for t in terminales:
        prod = tabla_ll1[nt].get(t, "")
        if prod == []:
            fila.append("ε")  # Producción vacía
        elif prod:
            fila.append(" ".join(prod))  # Mostrar producción como string
        else:
            fila.append("")  # Celda vacía
    print(f"{nt:<5}", "\t".join(fila))

# Reportar conflictos si los hay
if conflictos:
    print("\n⚠️ Conflictos detectados: la gramática NO es LL(1).")
    for nt, terminal, existente, nuevo in conflictos:
        print(f"  - Conflicto en [{nt}, {terminal}]: ya existe {existente}, se intenta agregar {nuevo}")
else:
    print("\n✅ Gramática confirmada como LL(1): no se detectaron conflictos en la tabla.")

def parse_ll1(entrada, tabla, simbolo_inicial):
    pila = ["$", simbolo_inicial]
    entrada = list(entrada) + ["$"]
    pasos = []

    while pila:
        tope = pila.pop()
        actual = entrada[0]

        pasos.append((list(pila), list(entrada)))

        if tope == actual:
            entrada.pop(0)
        elif tope not in tabla:
            return False, pasos
        elif actual in tabla[tope]:
            produccion = tabla[tope][actual]
            for simbolo in reversed(produccion):
                if simbolo != "":
                    pila.append(simbolo)
        else:
            return False, pasos

        if pila == ["$"] and entrada == ["$"]:
            pasos.append(([], ["$"]))
            return True, pasos
    return False, pasos


# Probar palabras
palabras = ["ab", "acb", "adb", "acdb", "a", "abbb"]
for palabra1 in palabras:
    palabra = list(palabra1)
    acepta, pasos = parse_ll1(palabra, tabla_ll1, "S")
    print("===========================================================")
    if acepta:
        print(f"✅ Palabra '{palabra1}' aceptada por el parser LL(1).\n")
        for i, (pila, entrada) in enumerate(pasos):
            print(f"Paso {i+1}: Pila = {pila}, Entrada = {entrada}")
    else:
        print(f"❌ Palabra '{palabra1}' no aceptada.")
        for i, (pila, entrada) in enumerate(pasos):
            print(f"Paso {i+1}: Pila = {pila}, Entrada = {entrada}")




📘 FIRST:
  S: {'a'}
  A: {'ε', 'c'}
  B: {'d', 'ε'}

📙 FOLLOW:
  S: {'$'}
  A: {'d', 'b'}
  B: {'b'}
📊 Tabla LL(1):

NT    $	a	b	c	d
S     	a A B b			
A     		ε	c	ε
B     		ε		d

✅ Gramática confirmada como LL(1): no se detectaron conflictos en la tabla.
✅ Palabra 'ab' aceptada por el parser LL(1).

Paso 1: Pila = ['$'], Entrada = ['a', 'b', '$']
Paso 2: Pila = ['$', 'b', 'B', 'A'], Entrada = ['a', 'b', '$']
Paso 3: Pila = ['$', 'b', 'B'], Entrada = ['b', '$']
Paso 4: Pila = ['$', 'b'], Entrada = ['b', '$']
Paso 5: Pila = ['$'], Entrada = ['b', '$']
Paso 6: Pila = [], Entrada = ['$']
✅ Palabra 'acb' aceptada por el parser LL(1).

Paso 1: Pila = ['$'], Entrada = ['a', 'c', 'b', '$']
Paso 2: Pila = ['$', 'b', 'B', 'A'], Entrada = ['a', 'c', 'b', '$']
Paso 3: Pila = ['$', 'b', 'B'], Entrada = ['c', 'b', '$']
Paso 4: Pila = ['$', 'b', 'B'], Entrada = ['c', 'b', '$']
Paso 5: Pila = ['$', 'b'], Entrada = ['b', '$']
Paso 6: Pila = ['$'], Entrada = ['b', '$']
Paso 7: Pila = [], Entrada = ['$']

#### **Ejercicio 2: Reflexión sobre el análisis**

**Descripción:**  
A partir de los resultados obtenidos en el ejercicio anterior, respondé en una celda de texto:

1. ¿Qué patrón tienen las palabras que fueron aceptadas?
2. ¿Qué tipo de errores aparecen en las palabras que fueron rechazadas?
3. ¿Cómo ayuda la tabla LL(1) a tomar decisiones en cada paso?
4. ¿Qué sucedería si quitamos la producción $A \to \varepsilon$? ¿Qué palabras dejarían de ser válidas?

**Objetivo:**  
Desarrollar intuición sobre el funcionamiento interno del parser LL(1) y el rol que cumple cada producción en el reconocimiento del lenguaje.



**¿Qué patrón tienen las palabras que fueron aceptadas?**
Las palabras que el parser da como validas comienzan con el caracter "a" y
terminan con el caracter "b" y tienen entre 2 y 4 caracteres.
Los caracteres intermedios entre "a" y "b" pueden ser "c", "d", o la
combinacion "cd"

**¿Qué tipo de errores aparecen en las palabras que fueron rechazadas?**
La primer palabra rechazada es "a" y no cumple con la "regla" de que cada
palabra debe comenzar con "a" y terminoar con "b".
La segunda palabra rechazada es "abbb" que si bien cumple la regla que comienza con "a" y termina con "b", no cumple con la validacion de los caracteres intermedios entre ambas que solo pueden ser "c", "d" o la combinacion "cd"

**¿Cómo ayuda la tabla LL(1) a tomar decisiones en cada paso?**

La tabla LL(1) es una estructura que guia la toma de decisiones en el análisis
sintáctico del algoritmo de parser.
Cada celda en dicha tabla indica qué producción se debe usar cuando el
"no terminal" *actual* (que aparece en la primer **columna** de la tabla) está en la parte superrior de la pila y el *símbolo actual* (ubicado en la primer **fila** de la tabla) es el simbolo de entrada que se está leyendo.

**¿Qué sucedería si quitamos la producción  A→ε ? ¿Qué palabras dejarían de ser válidas?**
En este caso las palabras validas serian palabras de 4 o 4 caracteres que comienzan por "a", luego viene una "c"; a continuacion podria haber una "d" (para palabras de 4 caracteres) o ε (para palabrs de 3 caracteres) y terminarian en "b"

Las unicas palabras vlidas de esa gramatica serian "acb" y "acdb"



#### **Ejercicio 3: Gramática con paréntesis balanceados**

**Descripción:**  
Vamos trabajar con una gramática clásica que reconoce expresiones con paréntesis correctamente balanceados. La estructura permite anidar o concatenar grupos de paréntesis.

**Gramática:**  
$$
\begin{aligned}
S &\rightarrow A \\
A &\rightarrow (A)A \mid \varepsilon
\end{aligned}
$$

**Objetivos:**  
- Construir la tabla LL(1) para esta gramática (usando Python).
- Analizar el comportamiento del parser frente a distintas cadenas de paréntesis (usando Python).
- Comprender cómo se representa la recursividad y opcionalidad en un análisis predictivo.

**Instrucciones:**

1. Reemplazá la gramática actual por esta:
```python
gramatica = {
    "S": [["A"]],
    "A": [["(", "A", ")", "A"], []]
}
```
2. Ejecutá las funciones `calcular_first`, `calcular_follow`, `construir_tabla_ll1`.
3. Verificá si hay conflictos y observá la tabla LL(1) generada.
4. Probá las siguientes palabras:
 - `""` (cadena vacía)
 - `"()"`
 - `"()()"`
 - `"(())"`
 - `"(()"`
 - `"())"`
 - `"(()())"`
5. Para cada palabra, usá `parse_ll1` para verificar si es aceptada.
6. En una celda de texto, respondé con sus palabras:
 - ¿Cuál es el patrón común en las palabras aceptadas?
  - ¿Por qué algunas palabras no se aceptan?
 - ¿Cómo ayuda la estructura de la gramática a asegurar el balanceo de paréntesis?

**Tip:**  
Podés representar las palabras como listas de caracteres:

```python
list("()()")
```

In [ ]:
# Gramática de ejemplo (LL(1))
gramatica = {
 "S": [["A"]],
 "A": [["(", "A", ")", "A"], []]
}

from collections import defaultdict

# Cálculo de FIRST
def calcular_first(gramatica):
    first = defaultdict(set)
    cambio = True
    while cambio:
        cambio = False
        for nt, producciones in gramatica.items():
            for prod in producciones:
                for simbolo in prod:
                    if simbolo not in gramatica:  # Terminal
                        if simbolo not in first[nt]:
                            first[nt].add(simbolo)
                            cambio = True
                        break
                    else:  # No terminal
                        nuevos = first[simbolo] - {"ε"}
                        if not nuevos.issubset(first[nt]):
                            first[nt].update(nuevos)
                            cambio = True
                        if "ε" not in first[simbolo]:
                            break
                else:
                    if "ε" not in first[nt]:
                        first[nt].add("ε")
                        cambio = True
    return dict(first)

# Cálculo de FOLLOW
def calcular_follow(gramatica, first, simbolo_inicial):
    follow = defaultdict(set)
    follow[simbolo_inicial].add("$")
    cambio = True
    while cambio:
        cambio = False
        for nt, producciones in gramatica.items():
            for prod in producciones:
                for i, simbolo in enumerate(prod):
                    if simbolo in gramatica:
                        siguiente = prod[i+1:]
                        first_siguiente = set()
                        for s in siguiente:
                            if s in gramatica:
                                first_siguiente |= (first[s] - {"ε"})
                                if "ε" not in first[s]:
                                    break
                            else:
                                first_siguiente.add(s)
                                break
                        else:
                            first_siguiente.add("ε")

                        if not first_siguiente - {"ε"} <= follow[simbolo]:
                            follow[simbolo] |= (first_siguiente - {"ε"})
                            cambio = True

                        if "ε" in first_siguiente or not siguiente:
                            if not follow[nt] <= follow[simbolo]:
                                follow[simbolo] |= follow[nt]
                                cambio = True
    return dict(follow)

# Mostrar resultados
first = calcular_first(gramatica)
follow = calcular_follow(gramatica, first, "S")

print("📘 FIRST:")
for nt, conj in first.items():
    print(f"  {nt}: {conj}")

print("\n📙 FOLLOW:")
for nt, conj in follow.items():
    print(f"  {nt}: {conj}")

# Esta función recorre todas las producciones de la gramática
# y extrae los símbolos terminales (los que no son no terminales).
# También agrega el símbolo $ al final, usado como marcador de fin de entrada.

def obtener_terminales(gramatica):
    terminales = set()
    for prods in gramatica.values():
        for prod in prods:
            for simbolo in prod:
                if simbolo not in gramatica:
                    terminales.add(simbolo)
    terminales.add("$")
    return sorted(terminales)


from collections import defaultdict

# Esta función construye la tabla LL(1) y detecta conflictos:
# Si una celda ya tiene una producción y se intenta colocar otra distinta,
# se registra como un conflicto.

def construir_tabla_ll1(gramatica, first, follow):
    tabla = defaultdict(dict)
    conflictos = []

    for nt, producciones in gramatica.items():
        for prod in producciones:
            first_prod = set()

            # Obtener el FIRST de esta producción
            for simbolo in prod:
                if simbolo not in gramatica:
                    first_prod.add(simbolo)
                    break
                else:
                    first_prod |= (first[simbolo] - {"ε"})
                    if "ε" not in first[simbolo]:
                        break
            else:
                first_prod.add("ε")

            # Parte 1: insertar la producción en la tabla según su FIRST
            for terminal in first_prod - {"ε"}:
                if terminal in tabla[nt]:
                    conflictos.append((nt, terminal, tabla[nt][terminal], prod))
                tabla[nt][terminal] = prod

            # Parte 2: si ε está en FIRST, usar FOLLOW para completar
            if "ε" in first_prod:
                for terminal in follow[nt]:
                    if terminal in tabla[nt]:
                        conflictos.append((nt, terminal, tabla[nt][terminal], prod))
                    tabla[nt][terminal] = []  # Representa A → ε

    return tabla, conflictos


# Generar la tabla
tabla_ll1, conflictos = construir_tabla_ll1(gramatica, first, follow)

# Obtener la lista de terminales de toda la gramática
terminales = obtener_terminales(gramatica)

# Mostrar encabezado
print("📊 Tabla LL(1):\n")
print(f"{'NT':<5}", "\t".join(terminales))

# Mostrar cada fila de la tabla
for nt in gramatica:
    fila = []
    for t in terminales:
        prod = tabla_ll1[nt].get(t, "")
        if prod == []:
            fila.append("ε")  # Producción vacía
        elif prod:
            fila.append(" ".join(prod))  # Mostrar producción como string
        else:
            fila.append("")  # Celda vacía
    print(f"{nt:<5}", "\t".join(fila))

# Reportar conflictos si los hay
if conflictos:
    print("\n⚠️ Conflictos detectados: la gramática NO es LL(1).")
    for nt, terminal, existente, nuevo in conflictos:
        print(f"  - Conflicto en [{nt}, {terminal}]: ya existe {existente}, se intenta agregar {nuevo}")
else:
    print("\n✅ Gramática confirmada como LL(1): no se detectaron conflictos en la tabla.")

def parse_ll1(entrada, tabla, simbolo_inicial):
    pila = ["$", simbolo_inicial]
    entrada = list(entrada) + ["$"]
    pasos = []

    while pila:
        tope = pila.pop()
        actual = entrada[0]

        pasos.append((list(pila), list(entrada)))

        if tope == actual:
            entrada.pop(0)
        elif tope not in tabla:
            return False, pasos
        elif actual in tabla[tope]:
            produccion = tabla[tope][actual]
            for simbolo in reversed(produccion):
                if simbolo != "":
                    pila.append(simbolo)
        else:
            return False, pasos

        if pila == ["$"] and entrada == ["$"]:
            pasos.append(([], ["$"]))
            return True, pasos
    return False, pasos


# Probar palabras
palabras = ["" , "()", "()()", "(())", "(()","())","(()())"]
for palabra1 in palabras:
    palabra = list(palabra1)
    acepta, pasos = parse_ll1(palabra, tabla_ll1, "S")
    print("===========================================================")
    if acepta:
        print(f"✅ Palabra '{palabra1}' aceptada por el parser LL(1).\n")
        for i, (pila, entrada) in enumerate(pasos):
            print(f"Paso {i+1}: Pila = {pila}, Entrada = {entrada}")
    else:
        print(f"❌ Palabra '{palabra1}' no aceptada.")
        for i, (pila, entrada) in enumerate(pasos):
            print(f"Paso {i+1}: Pila = {pila}, Entrada = {entrada}")


📘 FIRST:
  A: {'(', 'ε'}
  S: {'(', 'ε'}

📙 FOLLOW:
  S: {'$'}
  A: {')', '$'}
📊 Tabla LL(1):

NT    $	(	)
S     ε	A	
A     ε	( A ) A	ε

✅ Gramática confirmada como LL(1): no se detectaron conflictos en la tabla.
✅ Palabra '' aceptada por el parser LL(1).

Paso 1: Pila = ['$'], Entrada = ['$']
Paso 2: Pila = [], Entrada = ['$']
✅ Palabra '()' aceptada por el parser LL(1).

Paso 1: Pila = ['$'], Entrada = ['(', ')', '$']
Paso 2: Pila = ['$'], Entrada = ['(', ')', '$']
Paso 3: Pila = ['$', 'A', ')', 'A'], Entrada = ['(', ')', '$']
Paso 4: Pila = ['$', 'A', ')'], Entrada = [')', '$']
Paso 5: Pila = ['$', 'A'], Entrada = [')', '$']
Paso 6: Pila = ['$'], Entrada = ['$']
Paso 7: Pila = [], Entrada = ['$']
✅ Palabra '()()' aceptada por el parser LL(1).

Paso 1: Pila = ['$'], Entrada = ['(', ')', '(', ')', '$']
Paso 2: Pila = ['$'], Entrada = ['(', ')', '(', ')', '$']
Paso 3: Pila = ['$', 'A', ')', 'A'], Entrada = ['(', ')', '(', ')', '$']
Paso 4: Pila = ['$', 'A', ')'], Entrada = [')', '(', '

**¿Cuál es el patrón común en las palabras aceptadas?**

La gramática  propuesta define un lenguaje que reconoce cadenas de paréntesis
cuando estan correctamente balanceados. Es decir, aquellas palabras donde cada
paréntesis de apertura "(" tiene un paréntesis de cierre ")" correspondiente
y en el orden correcto.
Las palabras aceptadas son aquellas formadas por:
1) Cero o más pares de paréntesis balanceados.
2) Los paréntesis pueden estar anidados (uno dentro de otro), concatenados (uno después de otro), o combinaciones de ambas siempre que su estructura este balanceada.

**¿Por qué algunas palabras no se aceptan?**

Las palabras no se aceptan si:
1) El número de paréntesis de apertura "(" y de cierre ")" no es el mismo.
2) El orden es incorrecto, por ejemplo  **un ")" aparece antes de su correspondiente "("**.
3) Hay caracteres diferentes de "(" o ")"

**¿Cómo ayuda la estructura de la gramática a asegurar el balanceo de paréntesis?**

La gramática está diseñada para forzar el balanceo de paréntesis a traves de la regla **A → (A)A**

Esa regla asegura que:
1) Se abre un paréntesis (
2) Se genera una nueva cadena **A** que sera balanceada en cuanto a los parentesis ( esto lo hace recursivamente).
3) Se cierra el paréntesis ).
4) Se puede agregar otra subcadena balanceada **A**.

Cada vez que se abre un "(" la reglas de producción agrega un ")" después de un "No Terlminal" válido.

La estructura **(A)A** obliga que cada paréntesis abierto sea cerrado antes de continuar. De esta forma garantiza que "no se cierren paréntesis que no fueron abiertos" y que "cada apertura tenga su cierre".

En la produccion **(A)A**, la cadena **(A)** perimite el anidamiento de parentesis, mientras que la **A** final permite la concatenacion de los mismos.

